## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Model Training

For this exercice, it is necessary to have a model registered in MLFlow. For this we can, we can use the experiments from session 2.

In [6]:
pip install pandas scikit-learn mlflow

  Using cached mlflow-3.12.0-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-3.12.0-py3-none-any.whl.metadata (50 kB)
  Using cached mlflow_tracing-3.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached aiohttp-3.13.5-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.1 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.7-cp311-abi3-macosx_10_9_universal2.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached gunicorn-25.3.0-py3-none-any.whl.metadata (5.5 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached matplotlib-3.10.9-cp314-cp314-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached pandas-2.3.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (91 kB)
  Using

In [7]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import mlflow
from mlflow.models import infer_signature
...

Ellipsis

Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

Now, for the purpose of this exercice, you are required to define again the data transformation logic and save the one hot encoder as a `.pkl` file (if encoder was used during the pipeline).

In [8]:
file = "Churn_Modelling_train_test.csv"

In [ ]:
# Implement transformation logic as in session 2
def tranformation_logic(file):
    df = pd.read_csv(file)
    # Churn Dataset Cleaning and Correlation-Ready Pipeline
    # - Clean dataset for machine learning and analytics
    # - Prepare correlation-ready numerical dataset
    # - Handle missing values, duplicates, outliers, encoding
    # - Generate engineered features
    # - Export cleaned datasets
    # Strip spaces from column names immediately
    df.columns = df.columns.str.strip()
    # Remove duplicates
    df = df.drop_duplicates()
    # Standardize datatypes
    # Convert binary columns
    binary_cols = ['HasCrCard', 'IsActiveMember', 'Exited']

    for col in binary_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    # Missing values:
    # - Numerical -> median
    # - Categorical -> most frequent
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    categorical_cols = df.select_dtypes(include=['object']).columns

    # Numerical imputation
    num_imputer = SimpleImputer(strategy='median')
    df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

    # Categorical imputation
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

    # Clean string val
    for col in categorical_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.title()
        )

    # IQR for clipping outliers
    numeric_features = [
        col for col in numerical_cols
        if col != 'Exited'
    ]

    for col in numeric_features:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        df[col] = np.clip(df[col], lower_bound, upper_bound)

    # Create engineered variables in preparation for correlation analysis
# Balance-to-Salary Ratio
if {'Balance', 'EstimatedSalary'}.issubset(df.columns):
    df['BalanceSalaryRatio'] = (
        df['Balance'] / (df['EstimatedSalary'] + 1)
    )

# Age Groups
if 'Age' in df.columns:
    df['AgeGroup'] = pd.cut(
        df['Age'],
        bins=[0, 25, 35, 50, 65, 100],
        labels=['18-25', '26-35', '36-50', '51-65', '65+']
    )

# Product Engagement Score
if {'NumOfProducts', 'IsActiveMember'}.issubset(df.columns):
    df['EngagementScore'] = (
        df['NumOfProducts'] * df['IsActiveMember']
    )

# Copy before encoding
clean_df = df.copy()

# One-hot encoding for nominal categories
encoded_df = pd.get_dummies(
    clean_df,
    columns=['Geography', 'Gender', 'AgeGroup'],
    drop_first=True
)

# Try correlation
correlation_matrix = encoded_df.corr(numeric_only=True)

# Sort by correlation with target variable
if 'Exited' in correlation_matrix.columns:

    churn_corr = (
        correlation_matrix['Exited']
        .sort_values(ascending=False)
    )

    print('\n================ CORRELATION WITH CHURN ================')
    print(churn_corr)


# Export the clean dataset
# Main cleaned dataset
encoded_df.to_csv(
    'cleaned_churn_dataset.csv',
    index=False
)

# Correlation matrix
correlation_matrix.to_csv(
    'churn_correlation_matrix.csv'
)

# Descriptive analysis, Full statistical summary
stats_summary = encoded_df.describe(include='all').transpose()
print(stats_summary)

# Additional advanced metrics
advanced_stats = pd.DataFrame({
    'mean': encoded_df.mean(numeric_only=True),
    'median': encoded_df.median(numeric_only=True),
    'std_dev': encoded_df.std(numeric_only=True),
    'variance': encoded_df.var(numeric_only=True),
    'min': encoded_df.min(numeric_only=True),
    'max': encoded_df.max(numeric_only=True),
    'skewness': encoded_df.skew(numeric_only=True),
    'kurtosis': encoded_df.kurtosis(numeric_only=True)
})

In [ ]:
import joblib

PATH = ...
encoder = ...
joblib.dump(encoder, f'{PATH}one_hot_encoder.pkl')

In [ ]:
# Perform another experiment if you don't have the ones from session 2. Otherwise, this part can be skipped

# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

### Inference

In this part, you are asked to implement a function for batch and online inference methods by providing a model uri. 

In [ ]:
# import validation dataset to test inference
df_validation = pd.read_csv()

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [ ]:
# transform data - if necessary

##### Batch Inference

In [ ]:
# define a function to implement batch inference with mlflow
def batch_inference(model_uri: str, input: pd.DataFrame):
    pass

In [ ]:
# define the model uri that should be used
model_uri = ...

batch_prediction_result = batch_inference(model_uri, df_validation)

In [ ]:
# check the confusion matrix
from sklearn.metrics import confusion_matrix

##### Online Inference

For the online inference, it is required to set up local server. Follow the steps below to configure it:

1. Open a new bash terminal
2. Execute the follwing command `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080` in the terminal. You should specify the port that we are using for MLFlow
3. Execute the following command `mlflow models serve -m runs:/<run_id>/model -p 5000 --no-conda`. Note that `runs:/<run_id>/model` is your model uri.

In [ ]:
import requests
import json

In [ ]:
# import validation dataset to test inference - just one record
df_validation = pd.read_csv().head(1)

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [ ]:
# transform data - if necessary

In [4]:
def get_inference_endpoint(host="http://127.0.0.1", port=5000):
    return f"{host}:{port}/invocations"

url = get_inference_endpoint()

In [ ]:
# define a function to implement online inference with mlflow - pandas input
def online_inference_pandas(url: str, input: pd.DataFrame):
    ...

    response = requests.post()
    return response

In [ ]:
response_pandas = online_inference_pandas()
response_pandas.content

In [ ]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(url: str, input: dict):
    ...

    response = requests.post()
    return response

In [ ]:
# define the json as required by MLFlow
input_json = {
    "dataframe_split": {
        "columns": [],
        "data": [[], []]
    }
}

In [ ]:
response_json = online_inference_json()
response_json.content